In [26]:
!pip install -q sentence-transformers faiss-cpu langchain langchain-community datasets transformers accelerate bitsandbytes

In [27]:
!git clone https://github.com/abachaa/MedQuAD.git

fatal: destination path 'MedQuAD' already exists and is not an empty directory.


In [28]:
import os

root_dir = "MedQuAD"
for folder in sorted(os.listdir(root_dir))[:10]:
    print(folder)

.git
10_MPlus_ADAM_QA
11_MPlusDrugs_QA
12_MPlusHerbsSupplements_QA
1_CancerGov_QA
2_GARD_QA
3_GHR_QA
4_MPlus_Health_Topics_QA
5_NIDDK_QA
6_NINDS_QA


In [29]:
import xml.etree.ElementTree as ET
import glob

def parse_medquad_xml(filepath):
    """Extract (question, answer, focus/source) triples from one MedQuAD XML file."""
    qa_pairs = []
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()

        focus = root.findtext("Focus", default="")
        source = root.attrib.get("source", "")

        for qapair in root.iter("QAPair"):
            question = qapair.findtext("Question", default="").strip()
            answer = qapair.findtext("Answer", default="").strip()
            if question and answer:
                qa_pairs.append({
                    "focus": focus,
                    "source": source,
                    "question": question,
                    "answer": answer
                })
    except ET.ParseError:
        pass  # some files may be malformed; skip them
    return qa_pairs


# Walk through every XML file in every subfolder
all_files = glob.glob("MedQuAD/**/*.xml", recursive=True)
print(f"Found {len(all_files)} XML files")

all_qa = []
for f in all_files:
    all_qa.extend(parse_medquad_xml(f))

print(f"Total Q&A pairs extracted: {len(all_qa)}")
print(all_qa[0])

Found 11274 XML files
Total Q&A pairs extracted: 16407
{'focus': 'Lung agenesis', 'source': 'GARD', 'question': 'What are the symptoms of Lung agenesis ?', 'answer': 'What are the signs and symptoms of Lung agenesis? The Human Phenotype Ontology provides the following list of signs and symptoms for Lung agenesis. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Respiratory insufficiency 90% Abnormal lung lobation 50% Abnormality of the aorta 50% Anomalous pulmonary venous return 50% Aplasia/Hypoplasia of the lungs 50% Atria septal defect 50% Patent ductus arteriosus 50% Abnormality of the aortic valve 7.5% Abnormality of the helix 7.5% Abnormality of the ribs 7.5% Abnormality of the tricuspid valve 7.5% Aplasia/Hypoplasia of the thumb 7.5% Complete atrio

In [30]:
from datasets import Dataset

dataset = Dataset.from_list(all_qa)
print(dataset)
print(dataset[0])

Dataset({
    features: ['focus', 'source', 'question', 'answer'],
    num_rows: 16407
})
{'focus': 'Lung agenesis', 'source': 'GARD', 'question': 'What are the symptoms of Lung agenesis ?', 'answer': 'What are the signs and symptoms of Lung agenesis? The Human Phenotype Ontology provides the following list of signs and symptoms for Lung agenesis. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Respiratory insufficiency 90% Abnormal lung lobation 50% Abnormality of the aorta 50% Anomalous pulmonary venous return 50% Aplasia/Hypoplasia of the lungs 50% Atria septal defect 50% Patent ductus arteriosus 50% Abnormality of the aortic valve 7.5% Abnormality of the helix 7.5% Abnormality of the ribs 7.5% Abnormality of the tricuspid valve 7.5% Aplasia/Hypoplas

In [31]:
documents = []
metadata = []

for row in dataset:
    text = f"Question: {row['question']}\nAnswer: {row['answer']}"
    documents.append(text)
    metadata.append({"focus": row["focus"], "source": row["source"]})

print(f"Total documents: {len(documents)}")
print(documents[0])


Total documents: 16407
Question: What are the symptoms of Lung agenesis ?
Answer: What are the signs and symptoms of Lung agenesis? The Human Phenotype Ontology provides the following list of signs and symptoms for Lung agenesis. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Respiratory insufficiency 90% Abnormal lung lobation 50% Abnormality of the aorta 50% Anomalous pulmonary venous return 50% Aplasia/Hypoplasia of the lungs 50% Atria septal defect 50% Patent ductus arteriosus 50% Abnormality of the aortic valve 7.5% Abnormality of the helix 7.5% Abnormality of the ribs 7.5% Abnormality of the tricuspid valve 7.5% Aplasia/Hypoplasia of the thumb 7.5% Complete atrioventricular canal defect 7.5% Congenital diaphragmatic hernia 7.5% Preaxial hand poly

In [32]:
# Install embedding/retrieval libraries
!pip install -q sentence-transformers faiss-cpu

In [33]:
#Embed all documents
from sentence_transformers import SentenceTransformer
import numpy as np

# Medical-domain embedding model (better than generic ones for this data)
embed_model = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

embeddings = embed_model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(embeddings.shape)  # (num_docs, embedding_dim)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/257 [00:00<?, ?it/s]

(16407, 768)


In [34]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print(f"Indexed {index.ntotal} documents")

Indexed 16407 documents


In [35]:
def retrieve(query, k=3):
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_vec, k)

    results = []
    for idx in indices[0]:
        results.append({
            "text": documents[idx],
            "focus": metadata[idx]["focus"],
            "source": metadata[idx]["source"]
        })
    return results

# Quick test
results = retrieve("What are the symptoms of diabetes?")
for r in results:
    print(r["text"][:200])
    print("Source:", r["source"], "| Focus:", r["focus"])
    print("---")

Question: What are the symptoms of Diabetes ?
Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some los
Source: NIHSeniorHealth | Focus: Diabetes
---
Question: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2 ?
Answer: The signs and symptoms of diabetes are
                
- being very thirsty  - urinating often  - feeling very h
Source: NIDDK | Focus: Your Guide to Diabetes: Type 1 and Type 2
---
Question: What are the symptoms of Am I at Risk for Type 2 Diabetes? Taking Steps to Lower Your Risk of Getting Diabetes ?
Answer: The signs and symptoms of type 2 diabetes can be so mild that you mig
Source: NIDDK | Focus: Am I at Risk for Type 2 Diabetes? Taking Steps to Lower Your Risk of Getting Diabetes
---


In [36]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Ensure transformers and safetensors are up-to-date
!pip install --upgrade transformers safetensors

model_name = "google/flan-t5-large"  # fits free Colab; swap for a bigger model if you have Colab Pro/GPU

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")
gen_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

In [37]:
!pip install -q flask pyngrok


In [38]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN"  # paste yours here
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [39]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

In [40]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>MediAssist — AI Medical Q&A</title>
  <style>
    * { margin: 0; padding: 0; box-sizing: border-box; font-family: 'Segoe UI', Roboto, Arial, sans-serif; }
    body {
      background: linear-gradient(135deg, #e8f4f8 0%, #f0f7ff 100%);
      min-height: 100vh;
      display: flex;
      justify-content: center;
      padding: 40px 20px;
    }
    .container {
      max-width: 720px;
      width: 100%;
    }
    header {
      text-align: center;
      margin-bottom: 30px;
    }
    header h1 {
      color: #0b6e99;
      font-size: 2.2rem;
      display: flex;
      align-items: center;
      justify-content: center;
      gap: 10px;
    }
    header p {
      color: #5a6b73;
      margin-top: 8px;
      font-size: 0.95rem;
    }
    .disclaimer {
      background: #fff8e1;
      border-left: 4px solid #f5b400;
      padding: 12px 16px;
      border-radius: 6px;
      font-size: 0.85rem;
      color: #6b5a00;
      margin-bottom: 25px;
    }
    .chat-card {
      background: #fff;
      border-radius: 16px;
      box-shadow: 0 8px 30px rgba(0,0,0,0.08);
      padding: 25px;
    }
    .input-row {
      display: flex;
      gap: 10px;
      margin-bottom: 20px;
    }
    input[type="text"] {
      flex: 1;
      padding: 14px 16px;
      border: 1px solid #d8e2e6;
      border-radius: 10px;
      font-size: 1rem;
      outline: none;
      transition: border 0.2s;
    }
    input[type="text"]:focus {
      border-color: #0b6e99;
    }
    button {
      background: #0b6e99;
      color: white;
      border: none;
      padding: 0 24px;
      border-radius: 10px;
      font-size: 1rem;
      cursor: pointer;
      transition: background 0.2s;
    }
    button:hover { background: #095679; }
    button:disabled { background: #a9c3cc; cursor: not-allowed; }

    .answer-box {
      display: none;
      animation: fadeIn 0.3s ease-in;
    }
    @keyframes fadeIn {
      from { opacity: 0; transform: translateY(6px); }
      to { opacity: 1; transform: translateY(0); }
    }
    .answer-label {
      font-weight: 600;
      color: #0b6e99;
      margin-bottom: 8px;
      font-size: 0.9rem;
      text-transform: uppercase;
      letter-spacing: 0.5px;
    }
    .answer-text {
      background: #f4f9fb;
      padding: 16px 18px;
      border-radius: 10px;
      line-height: 1.6;
      color: #2b3a3f;
      white-space: pre-wrap;
    }
    .sources {
      margin-top: 16px;
    }
    .sources-label {
      font-size: 0.8rem;
      color: #7a8b91;
      text-transform: uppercase;
      letter-spacing: 0.5px;
      margin-bottom: 6px;
    }
    .source-chip {
      display: inline-block;
      background: #e8f2f6;
      color: #0b6e99;
      padding: 5px 12px;
      border-radius: 20px;
      font-size: 0.8rem;
      margin: 3px 4px 3px 0;
    }
    .loader {
      display: none;
      text-align: center;
      padding: 20px;
      color: #0b6e99;
      font-size: 0.9rem;
    }
    .spinner {
      border: 3px solid #e0eef2;
      border-top: 3px solid #0b6e99;
      border-radius: 50%;
      width: 28px;
      height: 28px;
      animation: spin 0.8s linear infinite;
      margin: 0 auto 10px;
    }
    @keyframes spin { to { transform: rotate(360deg); } }

    .examples {
      margin-top: 18px;
      display: flex;
      flex-wrap: wrap;
      gap: 8px;
    }
    .example-btn {
      background: #f4f9fb;
      color: #0b6e99;
      border: 1px solid #d8e2e6;
      padding: 8px 14px;
      border-radius: 20px;
      font-size: 0.82rem;
      cursor: pointer;
    }
    .example-btn:hover { background: #e8f2f6; }
  </style>
</head>
<body>
  <div class="container">
    <header>
      <h1>🩺 MediAssist</h1>
      <p>AI-powered medical information assistant (RAG-based, for research purposes only)</p>
    </header>

    <div class="disclaimer">
      ⚠️ This tool provides general medical information retrieved from public sources and is
      <strong>not a substitute for professional medical advice</strong>. Always consult a licensed
      healthcare provider for diagnosis or treatment.
    </div>

    <div class="chat-card">
      <div class="input-row">
        <input type="text" id="queryInput" placeholder="Ask a medical question, e.g. 'What are the symptoms of diabetes?'" />
        <button id="askBtn" onclick="askQuestion()">Ask</button>
      </div>

      <div class="examples">
        <span class="example-btn" onclick="fillExample(this)">What causes high blood pressure?</span>
        <span class="example-btn" onclick="fillExample(this)">What are the symptoms of asthma?</span>
        <span class="example-btn" onclick="fillExample(this)">How is anemia treated?</span>
      </div>

      <div class="loader" id="loader">
        <div class="spinner"></div>
        Searching medical knowledge base...
      </div>

      <div class="answer-box" id="answerBox" style="margin-top:20px;">
        <div class="answer-label">Answer</div>
        <div class="answer-text" id="answerText"></div>
        <div class="sources">
          <div class="sources-label">Sources</div>
          <div id="sourcesList"></div>
        </div>
      </div>
    </div>
  </div>

  <script>
    function fillExample(el) {
      document.getElementById('queryInput').value = el.textContent;
      askQuestion();
    }

    async function askQuestion() {
      const query = document.getElementById('queryInput').value.trim();
      if (!query) return;

      const askBtn = document.getElementById('askBtn');
      const loader = document.getElementById('loader');
      const answerBox = document.getElementById('answerBox');

      askBtn.disabled = true;
      loader.style.display = 'block';
      answerBox.style.display = 'none';

      try {
        const res = await fetch('/ask', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ query })
        });
        const data = await res.json();

        document.getElementById('answerText').textContent = data.answer;

        const sourcesList = document.getElementById('sourcesList');
        sourcesList.innerHTML = '';
        data.sources.forEach(s => {
          const chip = document.createElement('span');
          chip.className = 'source-chip';
          chip.textContent = s;
          sourcesList.appendChild(chip);
        });

        answerBox.style.display = 'block';
      } catch (err) {
        document.getElementById('answerText').textContent = 'Something went wrong. Please try again.';
        answerBox.style.display = 'block';
      } finally {
        loader.style.display = 'none';
        askBtn.disabled = false;
      }
    }

    document.getElementById('queryInput').addEventListener('keypress', function(e) {
      if (e.key === 'Enter') askQuestion();
    });
  </script>
</body>
</html>

Overwriting templates/index.html


In [41]:
from flask import Flask, request, jsonify, render_template

app = Flask(__name__)

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/ask", methods=["POST"])
def ask():
    data = request.get_json()
    query = data.get("query", "").strip()

    if not query:
        return jsonify({"answer": "Please enter a question.", "sources": []})

    answer, sources = rag_answer(query, k=3)  # uses your existing function
    unique_sources = list(set(sources))

    return jsonify({"answer": answer, "sources": unique_sources})

print("Flask app defined.")

Flask app defined.


In [42]:
from google.colab import userdata
from pyngrok import ngrok

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')  # <-- this string is the secret's NAME, always exactly this
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

port = 5000
public_url = ngrok.connect(port)
print("🌐 Your app is live at:", public_url)

import threading
def run_app():
    app.run(port=port)
thread = threading.Thread(target=run_app)
thread.start()

🌐 Your app is live at: NgrokTunnel: "https://exemption-buddhism-uptake.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
